In [1]:
import anndata as ad
import numpy as np
import pandas as pd
from importlib.metadata import version

for package in ("pandas", "anndata", "zarr"):
    package_version = version(package)
    print(f"{package}=={package_version}")


obs = pd.DataFrame(
    {
        "np_nan_str": [np.nan, "cell1"],
        "np_nan_int": [np.nan, 1],
        "np_nan_float": [np.nan, 1.0],
        "pd_NA_str": [pd.NA, "cell1"],
        "pd_NA_int": [pd.NA, 1],
        "pd_NA_float": [pd.NA, 1.0],
    },
    index=["cell1", "cell2"],
)

adata = ad.AnnData(
    X=np.zeros((2, 1)),
    obs=obs,
    var=pd.DataFrame(index=["gene1"]),
)

obs_dtypes_before_writing = adata.obs.dtypes.copy()

# print("\nObs before writing")
# print(adata.obs)

adata.write_zarr("toy_anndata.zarr")  # , convert_strings_to_categoricals=False)

print("\nDifferences in dtypes writing side-effect (no mention means no differences)")
print(obs_dtypes_before_writing.compare(adata.obs.dtypes, result_names=("Before writing", "After writing")))

reloaded = ad.read_zarr("toy_anndata.zarr")

# print("\nObs realoaded")
# print(reloaded.obs)

print("\nDifferences in Values round-trip (no mention means no differences)")
print(adata.obs.compare(reloaded.obs, result_names=("Before writing", "After reading")))


print("\nDifferences in dtypes round-trip (no mention means no differences)")
print(obs_dtypes_before_writing.compare(reloaded.obs.dtypes, result_names=("Before writing", "After reading")))

pandas==2.3.3
anndata==0.12.9
zarr==3.3.0


D:\Windows\repos\scverse\anndata\src\anndata\_io\zarr.py:44: UserWarning: Writing zarr v2 data will no longer be the default in the next minor release. v3 data will be written by default. If you are explicitly setting this configuration, consider migrating to the zarr v3 file format.
  f = open_write_group(store)



Differences in dtypes writing side-effect (no mention means no differences)
           Before writing After writing
np_nan_str         object      category
pd_NA_str          object      category

Differences in Values round-trip (no mention means no differences)
           pd_NA_int                  pd_NA_float              
      Before writing After reading Before writing After reading
cell1           <NA>          <NA>           <NA>          <NA>
cell2              1             1            1.0           1.0

Differences in dtypes round-trip (no mention means no differences)
           Before writing After reading
np_nan_str         object      category
pd_NA_str          object      category


In [4]:
import anndata as ad
import geopandas as gpd
import numpy as np
import pandas as pd
import spatialdata as sd
from shapely.geometry import Polygon
from spatialdata.models import ShapesModel, TableModel

for package in ("pandas", "anndata", "zarr"):
    package_version = version(package)
    print(f"{package}=={package_version}")

shapes = gpd.GeoDataFrame(
    {
        "geometry": [
            Polygon([(0, 0), (1, 0), (1, 1), (0, 1)]),
            Polygon([(2, 0), (3, 0), (3, 1), (2, 1)]),
        ]
    },
    index=["cell1", "cell2"],
)

obs = pd.DataFrame(
    {
        "instance_id": ["cell1", "cell2"],
        "region": ["cells", "cells"],
        "np_nan_str": [np.nan, "cell1"],
        "np_nan_int": [np.nan, 1],
        "np_nan_float": [np.nan, 1.0],
        "pd_NA_str": [pd.NA, "cell1"],
        "pd_NA_int": [pd.NA, 1],
        "pd_NA_float": [pd.NA, 1.0],
    },
    index=["cell1", "cell2"],
)

adata = ad.AnnData(
    X=np.zeros((2, 1)),
    obs=obs,
    var=pd.DataFrame(index=["gene1"]),
)

table = TableModel.parse(
    adata,
    region="cells",
    region_key="region",
    instance_key="instance_id",
)

sdata = sd.SpatialData(
    shapes={"cells": ShapesModel.parse(shapes)},
    tables={"counts": table},
)

sd_adata = sdata["counts"]

obs_dtypes_before_writing = sd_adata.obs.dtypes.copy()

# print("\nObs before writing")
# print(sd_adata.obs)

# print("\nObs dtypes before writing")
# print(obs_dtypes_before_writing)

sdata.write("toy_spatialdata.zarr", overwrite=True, convert_table_strings_to_categoricals=True)

# print("\nObs dtypes after writing")
# print(sd_adata.obs.dtypes)

print("\nDifferences in dtypes writing side-effect (no mention means no differences)")
print(obs_dtypes_before_writing.compare(sd_adata.obs.dtypes, result_names=("Before writing", "After writing")))

reloaded_sd = sd.read_zarr("toy_spatialdata.zarr")
reloaded_adata = reloaded_sd["counts"]

# print("\nObs realoaded")
# print(reloaded_adata.obs)

# print("\nObs dtypes after reading")
# print(reloaded_adata.obs.dtypes)

print("\nDifferences in Values round-trip (no mention means no differences)")
print(adata.obs.compare(reloaded_adata.obs, result_names=("Before writing", "After reading")))


print("\nDifferences in dtypes round-trip (no mention means no differences)")
print(obs_dtypes_before_writing.compare(reloaded_adata.obs.dtypes, result_names=("Before writing", "After reading")))

pandas==2.3.3
anndata==0.12.9
zarr==3.3.0


D:\Windows\repos\scverse\spatialdata_family\spatialdata\src\spatialdata\models\models.py:1267: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)



Differences in dtypes writing side-effect (no mention means no differences)
           Before writing After writing
np_nan_str         object      category
pd_NA_str          object      category

Differences in Values round-trip (no mention means no differences)
           pd_NA_int                  pd_NA_float              
      Before writing After reading Before writing After reading
cell1           <NA>          <NA>           <NA>          <NA>
cell2              1             1            1.0           1.0

Differences in dtypes round-trip (no mention means no differences)
           Before writing After reading
np_nan_str         object      category
pd_NA_str          object      category
